In [ ]:
"""Parsing and bucket labels for BEV instruction eval (text-only; no oracle annotation)."""

from __future__ import annotations

import re
from typing import Literal

DirectionBucket = Literal[
    "stop",
    "straight",
    "slight_left",
    "slight_right",
    "turn_left",
    "turn_right",
    "u_turn",
    "other",
]

SpeedBucket = Literal[
    "accelerating",
    "slowing_down",
    "constant_or_unspecified",
]

DIRECTION_BUCKETS: tuple[str, ...] = (
    "stop",
    "straight",
    "slight_left",
    "slight_right",
    "turn_left",
    "turn_right",
    "u_turn",
    "other",
)

SPEED_BUCKETS: tuple[str, ...] = (
    "accelerating",
    "slowing_down",
    "constant_or_unspecified",
)

INSTRUCTION_PROMPT = (
    "Based on the bird's-eye view of the driving scene, "
    "what driving instruction should the ego vehicle follow?"
)


def normalize_instruction(text: str) -> str:
    """Strict normalized string for exact-match eval."""
    normalized = str(text).lower().strip()
    normalized = re.sub(r"\s+", " ", normalized)
    normalized = normalized.rstrip(".,;:")
    return normalized


def direction_bucket(text: str) -> DirectionBucket:
    """Semantic direction bucket from instruction text only."""
    normalized = normalize_instruction(text)
    if not normalized:
        return "other"

    if normalized == "stop" or normalized.startswith("stop "):
        return "stop"

    if "u-turn" in normalized or "u turn" in normalized or "make a u-turn" in normalized:
        return "u_turn"

    if "slightly left" in normalized or "slight left" in normalized:
        return "slight_left"
    if "slightly right" in normalized or "slight right" in normalized:
        return "slight_right"

    if "turn left" in normalized or "to turn left" in normalized:
        return "turn_left"
    if "turn right" in normalized or "to turn right" in normalized:
        return "turn_right"

    if "changing lane to the left" in normalized or "change lane to the left" in normalized:
        return "slight_left"
    if "changing lane to the right" in normalized or "change lane to the right" in normalized:
        return "slight_right"

    if normalized.startswith("go left") or " go left" in normalized:
        return "turn_left"
    if normalized.startswith("go right") or " go right" in normalized:
        return "turn_right"

    if "go straight" in normalized or "following current lane" in normalized:
        return "straight"

    return "other"


def speed_bucket(text: str) -> SpeedBucket:
    """Speed / control bucket from instruction text only."""
    normalized = normalize_instruction(text)
    if "accelerat" in normalized:
        return "accelerating"
    if "slowing down" in normalized or "slow down" in normalized or "slowing" in normalized:
        return "slowing_down"
    return "constant_or_unspecified"


def direction_bucket_match(pred: str, target: str) -> bool:
    return direction_bucket(pred) == direction_bucket(target)


def speed_bucket_match(pred: str, target: str) -> bool:
    return speed_bucket(pred) == speed_bucket(target)


def semantic_bucket_match(pred: str, target: str) -> bool:
    return direction_bucket_match(pred, target) and speed_bucket_match(pred, target)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
from __future__ import annotations


import json
from dataclasses import fields
from pathlib import Path
from textwrap import shorten

import torch
import sys
sys.path.append("/zfsauton2/home/mineuih/waymax_rs/")
from data.inst_dataloader import build_inst_dataloader, resolve_cache_paths, split_cache_paths
from train_vla.configs.vla_finetuning_config import VLAFinetuningConfig
from train_vla.utils.utils import (
    build_vla_model,
    move_features_to_device,
    resolve_device,
    resolve_dtype,
    tokenize_text_batch,
)

CHECKPOINT_PATH = Path("/zfsauton/scratch/mineuih/waymax_rs/vla/finetune_vla/finetune_vla_gemma_20260527_030139/checkpoints/step_00040000.pt")
CACHE_DIR_OVERRIDE: str | None = None
INSTRUCTION_DIR_OVERRIDE: str | None = None
FILE_INDICES_OVERRIDE: list[int] | None = None
VALIDATION_FRACTION_OVERRIDE: float | None = None
GENERATE_SUBGOAL_OVERRIDE: bool | None = None
MAX_SAMPLES = 500
MAX_PROMPT_LENGTH = 128
MAX_ANSWER_LENGTH = 64
BATCH_SIZE = 20
NUM_WORKERS = 0
PIN_MEMORY = False
ADD_EOS_OVERRIDE: bool | None = None


def resolve_run_dir_and_checkpoint(path: Path) -> tuple[Path, Path]:
    path = path.expanduser().resolve()
    if path.is_file():
        if path.parent.name == "checkpoints":
            return path.parent.parent, path
        return path.parent, path
    if path.is_dir():
        if path.name == "checkpoints":
            checkpoint_files = sorted(path.glob("step_*.pt"))
            if not checkpoint_files:
                raise FileNotFoundError(f"No checkpoints found in {path}")
            return path.parent, checkpoint_files[-1]
        checkpoint_dir = path / "checkpoints"
        if checkpoint_dir.exists():
            checkpoint_files = sorted(checkpoint_dir.glob("step_*.pt"))
            if not checkpoint_files:
                raise FileNotFoundError(f"No checkpoints found in {checkpoint_dir}")
            return path, checkpoint_files[-1]
    raise FileNotFoundError(f"Could not resolve run directory from: {path}")


def load_run_config(run_dir: Path) -> dict:
    config_path = run_dir / "training_config.json"
    if not config_path.exists():
        raise FileNotFoundError(f"training_config.json not found: {config_path}")
    with config_path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def filter_finetuning_config(config_data: dict) -> dict:
    valid_keys = {field.name for field in fields(VLAFinetuningConfig)}
    return {key: value for key, value in config_data.items() if key in valid_keys}


def strip_module_prefix(state_dict: dict) -> dict:
    if not any(key.startswith("module.") for key in state_dict):
        return state_dict
    return {
        key[len("module.") :] if key.startswith("module.") else key: value
        for key, value in state_dict.items()
    }


def load_model_from_checkpoint(checkpoint_path: Path) -> tuple[torch.nn.Module, dict, Path, Path, torch.device, torch.dtype]:
    run_dir, resolved_checkpoint = resolve_run_dir_and_checkpoint(checkpoint_path)
    run_config = load_run_config(run_dir)
    model_config = filter_finetuning_config(run_config)
    model = build_vla_model(model_config)

    checkpoint = torch.load(resolved_checkpoint, map_location="cpu")
    state_dict = strip_module_prefix(checkpoint.get("model_state_dict", checkpoint))
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f"run_dir: {run_dir}")
    print(f"checkpoint: {resolved_checkpoint}")
    print(f"missing keys: {len(missing)}")
    print(f"unexpected keys: {len(unexpected)}")
    if missing:
        print("  missing:", missing[:10])
    if unexpected:
        print("  unexpected:", unexpected[:10])

    device = resolve_device()
    dtype = resolve_dtype(str(run_config.get("dtype", "bf16"))) if device.type == "cuda" else torch.float32
    model = model.to(device=device, dtype=dtype)
    model.eval()
    return model, run_config, run_dir, resolved_checkpoint, device, dtype


def resolve_eval_settings(run_config: dict) -> tuple[str, str, list[int] | None, int, bool, bool, float]:
    cache_dir = CACHE_DIR_OVERRIDE or str(run_config["cache_dir"])
    instruction_dir = INSTRUCTION_DIR_OVERRIDE or str(run_config["instruction_dir"])
    file_indices = FILE_INDICES_OVERRIDE if FILE_INDICES_OVERRIDE is not None else run_config.get("file_indices")
    anchor_step = int(run_config.get("anchor_step", 10))
    generate_subgoal = (
        GENERATE_SUBGOAL_OVERRIDE
        if GENERATE_SUBGOAL_OVERRIDE is not None
        else bool(run_config.get("generate_subgoal", False))
    )
    add_eos = ADD_EOS_OVERRIDE if ADD_EOS_OVERRIDE is not None else bool(run_config.get("add_eos", False))
    validation_fraction = (
        VALIDATION_FRACTION_OVERRIDE
        if VALIDATION_FRACTION_OVERRIDE is not None
        else float(run_config.get("validation_fraction", 0.2))
    )
    return cache_dir, instruction_dir, file_indices, anchor_step, generate_subgoal, add_eos, validation_fraction


def split_instruction_subgoal(model: torch.nn.Module, text: str) -> tuple[str, str]:
    splitter = getattr(model, "split_instruction_subgoal_text", None)
    if callable(splitter):
        return splitter(text)
    return text.strip(), ""


def generate_predictions_for_batch(
    model: torch.nn.Module,
    features: dict[str, torch.Tensor],
    prompt_ids: torch.Tensor,
    prompt_mask: torch.Tensor,
    max_new_tokens: int,
    generate_subgoal: bool,
) -> tuple[list[str], list[str]]:
    if generate_subgoal:
        generator = getattr(model, "generate_inst_subgoal_predictions", None)
        if callable(generator):
            return generator(
                input_features=features,
                prompt_ids=prompt_ids,
                prompt_mask=prompt_mask,
                max_new_tokens=max_new_tokens,
            )
    predictions = model.generate_predictions(
        input_features=features,
        prompt_ids=prompt_ids,
        prompt_mask=prompt_mask,
        max_new_tokens=max_new_tokens,
    )
    instruction_texts: list[str] = []
    subgoal_texts: list[str] = []
    for text in predictions:
        instruction_text, subgoal_text = split_instruction_subgoal(model, text)
        instruction_texts.append(instruction_text)
        subgoal_texts.append(subgoal_text)
    return instruction_texts, subgoal_texts


def inspect_validation_samples() -> None:
    model, run_config, _, _, device, dtype = load_model_from_checkpoint(CHECKPOINT_PATH)
    cache_dir, instruction_dir, file_indices, anchor_step, generate_subgoal, add_eos, validation_fraction = resolve_eval_settings(run_config)

    cache_paths = resolve_cache_paths(cache_dir, file_indices, anchor_step)
    train_cache_paths, val_cache_paths = split_cache_paths(cache_paths, validation_fraction)
    selected_cache_paths = val_cache_paths or train_cache_paths or cache_paths

    loader = build_inst_dataloader(
        cache_dir,
        anchor_step=anchor_step,
        file_indices=None,
        instruction_dir=instruction_dir,
        batch_size=BATCH_SIZE,
        shuffle_seed=0,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        cache_paths=selected_cache_paths,
        generate_subgoal=generate_subgoal,
    )

    print(f"device: {device}")
    print(f"dtype: {dtype}")
    print(f"cache_dir: {cache_dir}")
    print(f"instruction_dir: {instruction_dir}")
    print(f"validation_fraction: {validation_fraction}")
    print(f"generate_subgoal: {generate_subgoal}")
    print(f"num validation shards: {len(selected_cache_paths)}")

    seen = 0
    with torch.inference_mode():
        matching_accuracies = []
        direction_bucket_accuracies = []
        speed_bucket_accuracies = []
        semantic_bucket_accuracies = []
        for batch in loader:
            features, prompts, answers = batch.features, batch.prompts, batch.answers
            if not prompts:
                continue

            remaining = MAX_SAMPLES - seen
            if remaining <= 0:
                break
            if len(prompts) > remaining:
                prompts = prompts[:remaining]
                answers = answers[:remaining]
                features = {key: value[:remaining] for key, value in features.items()}

            prompt_ids, prompt_mask, _, _ = tokenize_text_batch(
                model.tokenizer,
                prompts,
                answers,
                add_eos=add_eos,
                device=device,
                max_prompt_length=MAX_PROMPT_LENGTH,
                max_answer_length=MAX_ANSWER_LENGTH,
            )
            features = move_features_to_device(features, device, dtype)

            amp_enabled = device.type == "cuda"
            with torch.autocast(device_type=device.type, dtype=dtype, enabled=amp_enabled):
                predicted_instructions, predicted_subgoals = generate_predictions_for_batch(
                    model=model,
                    features=features,
                    prompt_ids=prompt_ids,
                    prompt_mask=prompt_mask,
                    max_new_tokens=MAX_ANSWER_LENGTH,
                    generate_subgoal=generate_subgoal,
                )

            for idx, prompt in enumerate(prompts):
                actual_instruction, actual_subgoal = split_instruction_subgoal(model, answers[idx])
                predicted_instruction = predicted_instructions[idx]
                predicted_subgoal = predicted_subgoals[idx]
                # print("=" * 100)
                # print(f"sample {seen + 1}")
                # print(f"prompt: {prompt}")
                # print(f"predicted instruction: {shorten(predicted_instruction, width=300, placeholder='...')}")
                # if generate_subgoal:
                #     print(f"predicted subgoal: {predicted_subgoal}")
                # print(f"actual instruction: {shorten(actual_instruction, width=300, placeholder='...')}")
                # if generate_subgoal:
                #     print(f"actual subgoal: {actual_subgoal}")
                seen += 1
                if seen >= MAX_SAMPLES:
                    break
                predicted_instruction = normalize_instruction(predicted_instruction)
                actual_instruction = normalize_instruction(actual_instruction)
                matching_accuracies.append(predicted_instruction.strip().lower() == actual_instruction.strip().lower())
                direction_bucket_accuracies.append(direction_bucket_match(predicted_instruction, actual_instruction))
                speed_bucket_accuracies.append(speed_bucket_match(predicted_instruction, actual_instruction))
                semantic_bucket_accuracies.append(semantic_bucket_match(predicted_instruction, actual_instruction))
            if seen >= MAX_SAMPLES:
                break

    print(f"Matching accuracy: {sum(matching_accuracies) / len(matching_accuracies):.2%}")
    print(f"Direction bucket accuracy: {sum(direction_bucket_accuracies) / len(direction_bucket_accuracies):.2%}")
    print(f"Speed bucket accuracy: {sum(speed_bucket_accuracies) / len(speed_bucket_accuracies):.2%}")
    print(f"Semantic bucket accuracy: {sum(semantic_bucket_accuracies) / len(semantic_bucket_accuracies):.2%}")

inspect_validation_samples()


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

run_dir: /zfsauton/scratch/mineuih/waymax_rs/vla/finetune_vla/finetune_vla_gemma_20260527_030139
checkpoint: /zfsauton/scratch/mineuih/waymax_rs/vla/finetune_vla/finetune_vla_gemma_20260527_030139/checkpoints/step_00040000.pt
missing keys: 0
unexpected keys: 0
device: cuda
dtype: torch.bfloat16
cache_dir: /zfsauton/scratch/mineuih/waymax_rs/cache/
instruction_dir: /zfsauton/scratch/mineuih/waymax_rs/manual_instruction/
validation_fraction: 0.001
generate_subgoal: True
num validation shards: 1
Matching accuracy: 71.49%
